## Inter-Annotator Agreement (IAA) Analysis
for the suspense annotation study

Covers:
  - Ordinal columns: reader_suspense_level, character_anxiety_level
  - Categorical columns: character, suspense/anxiety-evoking_element
  - Free-text columns: arising_questions, question_answered_in_Sentence_ID

Metrics used:
  - Krippendorff's Alpha  (ordinal & nominal)
  - Fleiss' Kappa         (nominal / multi-rater)
  - Pairwise Cohen's Kappa
  - Spearman correlation  (ordinal pairwise)
  - Presence/absence agreement for sparse free-text fields
  - Cosine similarity (TF-IDF) for free-text content comparison

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from itertools import combinations
from collections import Counter
import re
import os

In [2]:
# ── statsmodels / sklearn / scipy ──────────────────────────────────────────────
from sklearn.metrics import cohen_kappa_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr, kendalltau


In [3]:
# ── Self-contained Krippendorff's Alpha ───────────────────────────────────────
def _krippendorff_alpha(reliability_data, level_of_measurement="ordinal"):
    """
    reliability_data: (n_raters x n_items) array-like, NaN = missing
    level_of_measurement: 'nominal' | 'ordinal' | 'interval' | 'ratio'
    """
    data = np.array(reliability_data, dtype=float)
    n_raters, n_items = data.shape

    # observed coincidences
    def coincidences(data):
        pairs = []
        for u in range(n_items):
            col = data[:, u]
            vals = col[~np.isnan(col)]
            m_u = len(vals)
            if m_u < 2:
                continue
            for i in range(len(vals)):
                for j in range(i + 1, len(vals)):
                    pairs.append((vals[i], vals[j], m_u))
        return pairs

    pairs = coincidences(data)
    if not pairs:
        return np.nan

    all_vals = np.concatenate([data[~np.isnan(data)]])
    unique_vals = np.unique(all_vals)

    def delta(c, k):
        if level_of_measurement == "nominal":
            return 0.0 if c == k else 1.0
        elif level_of_measurement in ("ordinal", "interval", "ratio"):
            # interval / ordinal use (c-k)^2
            return (c - k) ** 2
        return 0.0 if c == k else 1.0

    # observed disagreement Do
    D_o = sum(delta(c, k) / (m - 1) for c, k, m in pairs)

    # expected disagreement De (over all values weighted by marginals)
    n_total = sum(1 / (m - 1) for _, _, m in pairs)
    n_u_total = len(pairs) * 2 if pairs else 1   # approximate
    # marginal frequencies
    freq = Counter(all_vals)
    total_freq = sum(freq.values())

    D_e = 0.0
    val_list = list(freq.keys())
    for ci in val_list:
        for ck in val_list:
            D_e += freq[ci] * freq[ck] * delta(ci, ck)
    D_e = D_e / (total_freq * (total_freq - 1))

    # normalise D_o by total coincidences weight
    total_weight = sum(1.0 / (m - 1) for _, _, m in pairs)
    D_o_norm = D_o / total_weight if total_weight else 0

    if D_e == 0:
        return 1.0 if D_o_norm == 0 else 0.0
    return 1.0 - D_o_norm / D_e


class _KrippendorffModule:
    def alpha(self, reliability_data, level_of_measurement="ordinal"):
        return _krippendorff_alpha(reliability_data, level_of_measurement)

krippendorff = _KrippendorffModule()





In [4]:
# ── paths ──────────────────────────────────────────────────────────────────────
INPUT = "/Users/sguhr/Downloads/40321_Nesbit_Edith_The_mystery_of_the_semi_detached_3_annos.xlsx"
OUT_DIR = "/Users/sguhr/Desktop/URAP_Output"
os.makedirs(OUT_DIR, exist_ok=True)

PALETTE = sns.color_palette("Set2", 8)
SNS_STYLE = {"style": "whitegrid", "font_scale": 1.05}

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. LOAD DATA
# ══════════════════════════════════════════════════════════════════════════════

xl = pd.ExcelFile(INPUT)
raw = {sh: pd.read_excel(xl, sheet_name=sh) for sh in xl.sheet_names}

# Standardise column names once
COL_MAP = {
    "reader_suspense_level (0 = default; 1-5)":       "suspense",
    "character_anxiety_level (0 = default; 1-5)":     "anxiety",
    "character":                                       "character",
    "suspense/anxiety-evoking_element":                "element",
    "arising_questions (optional)":                    "questions",
    "question_answered_in_Sentence_ID":                "q_answered",
}

sheets = {}
for raw_name, df in raw.items():
    df = df.rename(columns=COL_MAP)
    # annotator first-name from sheet name
    annotator = raw_name.split("_")[0]
    df["annotator"] = annotator
    sheets[annotator] = df

annotators = list(sheets.keys())
print(f"Annotators ({len(annotators)}): {annotators}")
n_sentences = len(next(iter(sheets.values())))
print(f"Sentences per sheet: {n_sentences}")

# ── convenience: aligned matrix of a column across annotators ──────────────
def col_matrix(col):
    """Return (n_annotators x n_sentences) numpy array, NaN for missing."""
    return np.array([pd.to_numeric(sheets[a][col], errors="coerce").values for a in annotators], dtype=float)

Annotators (3): ['Irem', 'Hayden', 'Shannon']
Sentences per sheet: 68


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. HELPER: KRIPPENDORFF ALPHA
# ══════════════════════════════════════════════════════════════════════════════

def kripp_alpha(col, level_of_measurement):
    mat = col_matrix(col)
    try:
        alpha = krippendorff.alpha(reliability_data=mat,
                                   level_of_measurement=level_of_measurement)
    except Exception:
        alpha = np.nan
    return round(alpha, 4)

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. HELPER: FLEISS KAPPA (nominal, multi-rater)
# ══════════════════════════════════════════════════════════════════════════════

def fleiss_kappa_manual(mat_int, n_cat):
    """
    mat_int: (n_raters x n_items) integer array (NaN → -1 excluded from count)
    """
    n_items = mat_int.shape[1]
    n_raters = mat_int.shape[0]
    # build counts matrix (n_items x n_cat)
    counts = np.zeros((n_items, n_cat), dtype=float)
    for i in range(n_items):
        for r in range(n_raters):
            raw = mat_int[r, i]
            if np.isnan(raw): continue
            v = int(raw)
            if 0 <= v < n_cat:
                counts[i, v] += 1
    # items where at least 2 raters annotated
    n_j = counts.sum(axis=1)
    valid = n_j >= 2
    counts = counts[valid]
    n_j = n_j[valid]
    N = len(counts)
    if N == 0:
        return np.nan
    p_j = counts.sum(axis=0) / counts.sum()          # marginal proportions
    P_bar_e = (p_j ** 2).sum()
    P_i = ((counts ** 2).sum(axis=1) - n_j) / (n_j * (n_j - 1))
    P_bar = P_i.mean()
    if abs(1 - P_bar_e) < 1e-12:
        return np.nan
    kappa = (P_bar - P_bar_e) / (1 - P_bar_e)
    return round(kappa, 4)


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. PAIRWISE COHEN'S KAPPA & SPEARMAN
# ══════════════════════════════════════════════════════════════════════════════

def pairwise_cohen(col, weights=None):
    results = {}
    for a1, a2 in combinations(annotators, 2):
        s1 = pd.to_numeric(sheets[a1][col], errors="coerce").fillna(-1).astype(int)
        s2 = pd.to_numeric(sheets[a2][col], errors="coerce").fillna(-1).astype(int)
        mask = (s1 >= 0) & (s2 >= 0)
        if mask.sum() < 5:
            results[(a1, a2)] = np.nan
            continue
        try:
            k = cohen_kappa_score(s1[mask], s2[mask], weights=weights)
        except Exception:
            k = np.nan
        results[(a1, a2)] = round(k, 4)
    return results

def pairwise_spearman(col):
    results = {}
    for a1, a2 in combinations(annotators, 2):
        s1 = sheets[a1][col]
        s2 = sheets[a2][col]
        mask = s1.notna() & s2.notna()
        if mask.sum() < 5:
            results[(a1, a2)] = np.nan
            continue
        r, _ = spearmanr(s1[mask], s2[mask])
        results[(a1, a2)] = round(r, 4)
    return results

def pairwise_matrix(pw_dict):
    """Convert pairwise dict to symmetric n×n DataFrame."""
    mat = pd.DataFrame(index=annotators, columns=annotators, dtype=float)
    mat_arr = mat.values.copy(); np.fill_diagonal(mat_arr, 1.0); mat = pd.DataFrame(mat_arr, index=annotators, columns=annotators, dtype=float)
    for (a1, a2), v in pw_dict.items():
        mat.loc[a1, a2] = v
        mat.loc[a2, a1] = v
    return mat


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# 5. ORDINAL ANALYSIS: SUSPENSE & ANXIETY
# ══════════════════════════════════════════════════════════════════════════════

ordinal_cols = ["suspense", "anxiety"]
ordinal_results = {}

for col in ordinal_cols:
    alpha_ord  = kripp_alpha(col, "ordinal")
    alpha_int  = kripp_alpha(col, "interval")
    fleiss_k   = fleiss_kappa_manual(col_matrix(col), n_cat=6)
    pw_kappa   = pairwise_cohen(col, weights="quadratic")
    pw_spear   = pairwise_spearman(col)
    ordinal_results[col] = {
        "kripp_ordinal":  alpha_ord,
        "kripp_interval": alpha_int,
        "fleiss_kappa":   fleiss_k,
        "pw_kappa":       pw_kappa,
        "pw_spearman":    pw_spear,
    }
    print(f"\n{col.upper()}")
    print(f"  Krippendorff α (ordinal):  {alpha_ord}")
    print(f"  Krippendorff α (interval): {alpha_int}")
    print(f"  Fleiss κ:                  {fleiss_k}")


SUSPENSE
  Krippendorff α (ordinal):  0.1583
  Krippendorff α (interval): 0.1583
  Fleiss κ:                  0.0088

ANXIETY
  Krippendorff α (ordinal):  0.0404
  Krippendorff α (interval): 0.0404
  Fleiss κ:                  -0.0403


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# 6. CATEGORICAL ANALYSIS: CHARACTER
# ══════════════════════════════════════════════════════════════════════════════

# Encode character labels to integers
def encode_col(col):
    all_vals = pd.concat([sheets[a][col] for a in annotators]).dropna().unique()
    mapping = {v: i for i, v in enumerate(sorted(all_vals))}
    encoded = {}
    for a in annotators:
        encoded[a] = sheets[a][col].map(mapping).values  # NaN for missing
    return mapping, np.array([encoded[a] for a in annotators], dtype=float)

char_mapping, char_mat = encode_col("character")
alpha_char  = krippendorff.alpha(reliability_data=char_mat, level_of_measurement="nominal")
fleiss_char = fleiss_kappa_manual(char_mat, n_cat=len(char_mapping))
pw_char     = pairwise_cohen("character".replace("character", "character"), weights=None)

# need int-encoded series for pairwise cohen
def pairwise_cohen_cat(col):
    mapping, _ = encode_col(col)
    results = {}
    for a1, a2 in combinations(annotators, 2):
        s1 = sheets[a1][col].map(mapping).fillna(-1).astype(int)
        s2 = sheets[a2][col].map(mapping).fillna(-1).astype(int)
        mask = (s1 >= 0) & (s2 >= 0)
        if mask.sum() < 5:
            results[(a1, a2)] = np.nan; continue
        try:
            k = cohen_kappa_score(s1[mask], s2[mask])
        except Exception:
            k = np.nan
        results[(a1, a2)] = round(k, 4)
    return results

pw_char_kappa = pairwise_cohen_cat("character")

print(f"\nCHARACTER")
print(f"  Krippendorff α (nominal): {round(alpha_char, 4)}")
print(f"  Fleiss κ:                 {fleiss_char}")



CHARACTER
  Krippendorff α (nominal): -0.6717
  Fleiss κ:                 -0.5038


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# 7. FREE-TEXT: ELEMENT, QUESTIONS
# ══════════════════════════════════════════════════════════════════════════════

def presence_agreement(col):
    """% sentences where all annotators agree on presence/absence."""
    pres = pd.DataFrame({a: sheets[a][col].notna().astype(int) for a in annotators})
    full_agree = (pres.nunique(axis=1) == 1).mean()
    pres_rate  = {a: pres[a].mean() for a in annotators}
    return round(full_agree, 4), pres_rate

def pairwise_tfidf_similarity(col):
    """Average pairwise cosine similarity of TF-IDF vectors for non-empty cells."""
    sims = {}
    for a1, a2 in combinations(annotators, 2):
        texts_a1 = sheets[a1][col].fillna("").tolist()
        texts_a2 = sheets[a2][col].fillna("").tolist()
        combined = texts_a1 + texts_a2
        if all(t == "" for t in combined):
            sims[(a1, a2)] = np.nan; continue
        try:
            vect = TfidfVectorizer(min_df=1)
            mat  = vect.fit_transform(combined)
            m1 = mat[:len(texts_a1)]
            m2 = mat[len(texts_a1):]
            # per-sentence cosine where both annotated
            scores = []
            for i in range(len(texts_a1)):
                if texts_a1[i] and texts_a2[i]:
                    s = cosine_similarity(m1[i], m2[i])[0, 0]
                    scores.append(s)
            sims[(a1, a2)] = round(np.mean(scores), 4) if scores else np.nan
        except Exception:
            sims[(a1, a2)] = np.nan
    return sims

print("\nFREE-TEXT fields:")
for col in ["element", "questions"]:
    agree, rates = presence_agreement(col)
    sims = pairwise_tfidf_similarity(col)
    print(f"  {col}: presence-agreement={agree}, avg-tfidf-sim={np.nanmean(list(sims.values())):.3f}")



FREE-TEXT fields:
  element: presence-agreement=0.3529, avg-tfidf-sim=0.018
  questions: presence-agreement=0.9853, avg-tfidf-sim=nan


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# 8. PER-SENTENCE MEAN SCORES
# ══════════════════════════════════════════════════════════════════════════════

ref = next(iter(sheets.values()))
sent_df = pd.DataFrame({"Sentence_ID": ref["Sentence_ID"], "Sentence": ref["Sentence"]})
for col in ["suspense", "anxiety"]:
    for a in annotators:
        sent_df[f"{col}_{a}"] = pd.to_numeric(sheets[a][col], errors="coerce").values
    sent_df[f"{col}_mean"] = sent_df[[f"{col}_{a}" for a in annotators]].mean(axis=1)
    sent_df[f"{col}_std"]  = sent_df[[f"{col}_{a}" for a in annotators]].std(axis=1)

In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# 9. VISUALISATIONS
# ══════════════════════════════════════════════════════════════════════════════

sns.set(**SNS_STYLE)
COLOR_SUSPENSE = "#E07B54"
COLOR_ANXIETY  = "#5B8DB8"


In [14]:
# ─── Fig 1: Per-sentence mean + individual lines ───────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
for ax, col, color, label in zip(
    axes,
    ["suspense", "anxiety"],
    [COLOR_SUSPENSE, COLOR_ANXIETY],
    ["Reader Suspense Level", "Character Anxiety Level"]
):
    x = sent_df["Sentence_ID"]
    for i, a in enumerate(annotators):
        ax.plot(x, sent_df[f"{col}_{a}"], alpha=0.35, linewidth=1.2,
                color=PALETTE[i], label=a)
    ax.plot(x, sent_df[f"{col}_mean"], color=color, linewidth=2.5,
            label="Mean", zorder=5)
    ax.fill_between(x,
                    sent_df[f"{col}_mean"] - sent_df[f"{col}_std"],
                    sent_df[f"{col}_mean"] + sent_df[f"{col}_std"],
                    alpha=0.15, color=color, label="±1 SD")
    ax.set_ylabel(label, fontsize=11)
    ax.set_ylim(-0.2, 5.5)
    ax.legend(fontsize=8, ncol=len(annotators)+2, loc="upper right")
    ax.set_title(f"{label} — all annotators + mean ± SD", fontsize=12)

axes[-1].set_xlabel("Sentence ID", fontsize=11)
plt.tight_layout()
fig.savefig(f"{OUT_DIR}/fig1_per_sentence_scores.png", dpi=150)
plt.close()
print("Saved fig1")

Saved fig1


In [15]:
# ─── Fig 2a: Heatmap of individual scores ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, col, title in zip(axes, ["suspense", "anxiety"],
                           ["Reader Suspense", "Character Anxiety"]):
    data = pd.DataFrame({a: pd.to_numeric(sheets[a][col], errors="coerce").values for a in annotators},
                        index=ref["Sentence_ID"])
    cmap = LinearSegmentedColormap.from_list("", ["#f7f7f7", "#d95f02" if col=="suspense" else "#1f78b4"])
    sns.heatmap(data.T, ax=ax, cmap=cmap, vmin=0, vmax=5,
                linewidths=0.3, linecolor="#ddd",
                cbar_kws={"label": "Score (0–5)"})
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Sentence ID")
    ax.set_ylabel("Annotator")
plt.tight_layout()
fig.savefig(f"{OUT_DIR}/fig2a_annotation_heatmaps.png", dpi=150)
plt.close()
print("Saved fig2")

Saved fig2


In [16]:
# ─── Fig 2b: Heatmap of individual scores (annotators on x, sentences on y) ──
fig, axes = plt.subplots(1, 2, figsize=(10, 18))
for ax, col, title in zip(axes, ["suspense", "anxiety"],
                           ["Reader Suspense", "Character Anxiety"]):
    data = pd.DataFrame(
        {a: pd.to_numeric(sheets[a][col], errors="coerce").values for a in annotators},
        index=ref["Sentence_ID"]
    )  # rows = sentences, columns = annotators — no transpose needed
    cmap = LinearSegmentedColormap.from_list(
        "", ["#f7f7f7", "#d95f02" if col == "suspense" else "#1f78b4"]
    )
    sns.heatmap(data, ax=ax, cmap=cmap, vmin=0, vmax=5,
                linewidths=0.3, linecolor="#ddd",
                cbar_kws={"label": "Score (0–5)"})
    ax.set_title(title, fontsize=13)
    ax.set_ylabel("Sentence ID")
    ax.set_xlabel("Annotator")
plt.tight_layout()
fig.savefig(f"{OUT_DIR}/fig2b_annotation_heatmaps.png", dpi=150)
plt.close()

In [17]:
# ─── Fig 3: Pairwise agreement matrices ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ["Suspense — Quadratic κ", "Anxiety — Quadratic κ", "Character — κ"]
mats_to_plot = [
    pairwise_matrix(ordinal_results["suspense"]["pw_kappa"]),
    pairwise_matrix(ordinal_results["anxiety"]["pw_kappa"]),
    pairwise_matrix(pw_char_kappa),
]
for ax, mat, title in zip(axes, mats_to_plot, titles):
    mask = np.eye(len(annotators), dtype=bool)
    sns.heatmap(mat.astype(float), ax=ax, annot=True, fmt=".2f",
                cmap="RdYlGn", vmin=-0.3, vmax=1.0,
                linewidths=0.5, square=True, mask=mask,
                cbar_kws={"shrink": 0.75})
    ax.set_title(title, fontsize=11)
plt.suptitle("Pairwise Agreement Matrices", fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(f"{OUT_DIR}/fig3_pairwise_agreement.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved fig3")

Saved fig3


In [18]:
# ─── Fig 4: Spearman pairwise matrices ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, title in zip(axes, ["suspense", "anxiety"],
                           ["Suspense — Spearman ρ", "Anxiety — Spearman ρ"]):
    mat = pairwise_matrix(ordinal_results[col]["pw_spearman"])
    mask = np.eye(len(annotators), dtype=bool)
    sns.heatmap(mat.astype(float), ax=ax, annot=True, fmt=".2f",
                cmap="coolwarm", vmin=-1, vmax=1,
                linewidths=0.5, square=True, mask=mask,
                cbar_kws={"shrink": 0.75})
    ax.set_title(title, fontsize=11)
plt.suptitle("Pairwise Spearman Correlations", fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(f"{OUT_DIR}/fig4_spearman_heatmaps.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved fig4")

Saved fig4


In [19]:
# ─── Fig 5: Score distributions ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, title, color in zip(
    axes, ["suspense", "anxiety"],
    ["Reader Suspense Distribution", "Character Anxiety Distribution"],
    [COLOR_SUSPENSE, COLOR_ANXIETY]
):
    all_scores = pd.concat([sheets[a][[col, "annotator"]].assign(**{col: pd.to_numeric(sheets[a][col], errors="coerce")}) for a in annotators], ignore_index=True)
    sns.histplot(data=all_scores, x=col, hue="annotator",
                 multiple="dodge", discrete=True, ax=ax,
                 palette=PALETTE[:len(annotators)], shrink=0.8)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Score (0 = none, 1–5 = intensity)")
    ax.set_ylabel("Count")
plt.tight_layout()
fig.savefig(f"{OUT_DIR}/fig5_score_distributions.png", dpi=150)
plt.close()
print("Saved fig5")

Saved fig5


In [20]:
# ─── Fig 6: High-disagreement sentences ─────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=False)
for ax, col, color, title in zip(
    axes, ["suspense", "anxiety"],
    [COLOR_SUSPENSE, COLOR_ANXIETY],
    ["Suspense std dev per sentence", "Anxiety std dev per sentence"]
):
    std_vals = sent_df[f"{col}_std"]
    bars = ax.bar(sent_df["Sentence_ID"], std_vals, color=color, alpha=0.75)
    # highlight top-5 disagreements
    top5 = std_vals.nlargest(5).index
    for idx in top5:
        bars[idx].set_edgecolor("black")
        bars[idx].set_linewidth(1.5)
        ax.text(sent_df["Sentence_ID"][idx], std_vals[idx] + 0.03,
                str(sent_df["Sentence_ID"][idx]), ha="center", fontsize=8)
    ax.set_ylabel("Std Dev")
    ax.set_title(title, fontsize=12)
    ax.set_ylim(0, 3)
axes[-1].set_xlabel("Sentence ID")
plt.tight_layout()
fig.savefig(f"{OUT_DIR}/fig6_disagreement_by_sentence.png", dpi=150)
plt.close()
print("Saved fig6")

Saved fig6


In [21]:
# ─── Fig 7: TF-IDF pairwise similarity for free-text ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, title in zip(axes, ["element", "questions"],
                           ["Suspense Element (TF-IDF cosine sim)",
                            "Arising Questions (TF-IDF cosine sim)"]):
    sims = pairwise_tfidf_similarity(col)
    mat = pairwise_matrix(sims)
    mask = np.eye(len(annotators), dtype=bool)
    sns.heatmap(mat.astype(float), ax=ax, annot=True, fmt=".2f",
                cmap="YlOrRd", vmin=0, vmax=1,
                linewidths=0.5, square=True, mask=mask,
                cbar_kws={"shrink": 0.75})
    ax.set_title(title, fontsize=11)
plt.suptitle("Pairwise Text Similarity (TF-IDF cosine) for Free-Text Fields", fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(f"{OUT_DIR}/fig7_freetext_similarity.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved fig7")

Saved fig7


In [22]:
# ─── Fig 8: Character label frequency per annotator ─────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
char_counts = pd.DataFrame({
    a: sheets[a]["character"].value_counts() for a in annotators
}).fillna(0).astype(int)
char_counts.T.plot(kind="bar", ax=ax, colormap="Set2", width=0.75)
ax.set_title("Character Label Frequency per Annotator", fontsize=12)
ax.set_xlabel("Annotator")
ax.set_ylabel("Count")
ax.legend(title="Character", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
fig.savefig(f"{OUT_DIR}/fig8_character_frequency.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved fig8")

Saved fig8


In [23]:
# ─── Fig 9: Presence rate for free-text fields ──────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
pres_data = {}
for col in ["element", "questions", "q_answered"]:
    _, rates = presence_agreement(col)
    pres_data[col] = rates

pres_df = pd.DataFrame(pres_data, index=annotators)
pres_df.plot(kind="bar", ax=ax, colormap="Accent", width=0.65)
ax.set_title("Annotation Presence Rate per Annotator & Field", fontsize=12)
ax.set_ylabel("Proportion of sentences annotated")
ax.set_xlabel("Annotator")
ax.set_xticklabels(annotators, rotation=15)
ax.legend(title="Field", labels=["Suspense Element", "Arising Questions", "Q Answered (ID)"])
plt.tight_layout()
fig.savefig(f"{OUT_DIR}/fig9_presence_rates.png", dpi=150)
plt.close()
print("Saved fig9")

Saved fig9


In [24]:
# ─── Fig 10: Summary IAA bar chart ──────────────────────────────────────────
summary_labels = [
    "Suspense\nKripp α (ord.)",
    "Suspense\nFleiss κ",
    "Anxiety\nKripp α (ord.)",
    "Anxiety\nFleiss κ",
    "Character\nKripp α (nom.)",
    "Character\nFleiss κ",
]
summary_values = [
    ordinal_results["suspense"]["kripp_ordinal"],
    ordinal_results["suspense"]["fleiss_kappa"],
    ordinal_results["anxiety"]["kripp_ordinal"],
    ordinal_results["anxiety"]["fleiss_kappa"],
    round(alpha_char, 4),
    fleiss_char,
]
colors_bar = [COLOR_SUSPENSE, COLOR_SUSPENSE, COLOR_ANXIETY, COLOR_ANXIETY, "#6a9e5b", "#6a9e5b"]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(summary_labels, summary_values, color=colors_bar, alpha=0.85, edgecolor="white", linewidth=1.2)
# threshold lines
for y, lbl, ls in [(0.8, "Strong (0.8)", "--"), (0.6, "Moderate (0.6)", ":"), (0.2, "Fair (0.2)", "-.")]:
    ax.axhline(y, color="gray", linestyle=ls, linewidth=0.9, alpha=0.7)
    ax.text(5.6, y + 0.01, lbl, fontsize=8, color="gray", ha="right")
for bar, val in zip(bars, summary_values):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_ylim(-0.1, 1.1)
ax.set_ylabel("Agreement Score")
ax.set_title("Summary: Inter-Annotator Agreement Metrics", fontsize=13)
plt.tight_layout()
fig.savefig(f"{OUT_DIR}/fig10_iaa_summary_bar.png", dpi=150)
plt.close()
print("Saved fig10")

Saved fig10


In [25]:
# ══════════════════════════════════════════════════════════════════════════════
# 10. QUALITATIVE: TOP DISAGREEMENT SENTENCES
# ══════════════════════════════════════════════════════════════════════════════

disagreement_rows = []
for col in ["suspense", "anxiety"]:
    top = sent_df.nlargest(5, f"{col}_std")[
        ["Sentence_ID", "Sentence", f"{col}_mean", f"{col}_std"]
        + [f"{col}_{a}" for a in annotators]
    ].copy()
    top.columns = (["Sentence_ID", "Sentence", "Mean", "StdDev"]
                   + [f"Score_{a}" for a in annotators])
    top.insert(0, "Column", col)
    disagreement_rows.append(top)

disagreement_df = pd.concat(disagreement_rows, ignore_index=True)

In [26]:
# ══════════════════════════════════════════════════════════════════════════════
# 11. EXPORT: SUMMARY STATS TABLE
# ══════════════════════════════════════════════════════════════════════════════

summary_rows = []
for col, label in [("suspense", "Reader Suspense"), ("anxiety", "Character Anxiety")]:
    r = ordinal_results[col]
    summary_rows.append({
        "Field": label,
        "Type": "Ordinal (0–5)",
        "Kripp α (ordinal)":  r["kripp_ordinal"],
        "Kripp α (interval)": r["kripp_interval"],
        "Fleiss κ":           r["fleiss_kappa"],
        "Mean pairwise κ (quadratic)": round(np.nanmean(list(r["pw_kappa"].values())), 4),
        "Mean pairwise Spearman ρ":    round(np.nanmean(list(r["pw_spearman"].values())), 4),
    })

summary_rows.append({
    "Field": "Character",
    "Type": "Nominal",
    "Kripp α (ordinal)":  round(alpha_char, 4),
    "Kripp α (interval)": "–",
    "Fleiss κ":           fleiss_char,
    "Mean pairwise κ (quadratic)": round(np.nanmean(list(pw_char_kappa.values())), 4),
    "Mean pairwise Spearman ρ": "–",
})

for col, label in [("element", "Suspense Element"), ("questions", "Arising Questions")]:
    agree, _ = presence_agreement(col)
    sims = pairwise_tfidf_similarity(col)
    summary_rows.append({
        "Field": label,
        "Type": "Free text",
        "Kripp α (ordinal)": "–",
        "Kripp α (interval)": "–",
        "Fleiss κ": "–",
        "Mean pairwise κ (quadratic)": "–",
        "Mean pairwise Spearman ρ": "–",
        "Presence agreement": agree,
        "Mean TF-IDF cosine sim": round(np.nanmean(list(sims.values())), 4),
    })

summary_table = pd.DataFrame(summary_rows).fillna("–")

In [27]:
# ══════════════════════════════════════════════════════════════════════════════
# 12. EXPORT CSV TABLES
# ══════════════════════════════════════════════════════════════════════════════

summary_table.to_csv(f"{OUT_DIR}/table_iaa_summary.csv", index=False)

# Per-sentence mean/std
sent_export = sent_df[
    ["Sentence_ID", "Sentence"]
    + [f"suspense_{a}" for a in annotators]
    + ["suspense_mean", "suspense_std"]
    + [f"anxiety_{a}" for a in annotators]
    + ["anxiety_mean", "anxiety_std"]
]
sent_export.to_csv(f"{OUT_DIR}/table_per_sentence_scores.csv", index=False)

# Pairwise kappa tables
for col, label, pw in [
    ("suspense", "suspense_kappa", ordinal_results["suspense"]["pw_kappa"]),
    ("anxiety",  "anxiety_kappa",  ordinal_results["anxiety"]["pw_kappa"]),
    ("character","character_kappa",pw_char_kappa),
]:
    pairwise_matrix(pw).to_csv(f"{OUT_DIR}/table_{label}_pairwise.csv")

# Disagreement table
disagreement_df.to_csv(f"{OUT_DIR}/table_top_disagreements.csv", index=False)

In [28]:
# ══════════════════════════════════════════════════════════════════════════════
# 13. PRINT FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═"*60)
print("IAA SUMMARY TABLE")
print("═"*60)
print(summary_table.to_string(index=False))

print("\n" + "═"*60)
print("TOP 5 DISAGREEMENT SENTENCES — SUSPENSE")
print("═"*60)
top_susp = disagreement_df[disagreement_df.Column=="suspense"][["Sentence_ID","Mean","StdDev"]
    + [f"Score_{a}" for a in annotators]].to_string(index=False)
print(top_susp)

print("\nDone. Files written to", OUT_DIR)


════════════════════════════════════════════════════════════
IAA SUMMARY TABLE
════════════════════════════════════════════════════════════
            Field          Type Kripp α (ordinal) Kripp α (interval) Fleiss κ Mean pairwise κ (quadratic) Mean pairwise Spearman ρ Presence agreement Mean TF-IDF cosine sim
  Reader Suspense Ordinal (0–5)            0.1583             0.1583   0.0088                      0.1716                   0.1399                  –                      –
Character Anxiety Ordinal (0–5)            0.0404             0.0404  -0.0403                      0.0636                   0.1142                  –                      –
        Character       Nominal           -0.6717                  –  -0.5038                         0.0                        –                  –                      –
 Suspense Element     Free text                 –                  –        –                           –                        –             0.3529                 0